# Analysis of EuRepoC Reported Cyber Incidents 
## Goal is to use NLP and machine learning to predict target countries of cyber attacks

**Aim 1**: Using the the latent topics extracted from incident descriptions (see build_BERTopic_model.ipynb), are specific types of cyber attacks linked to differences in terms of societal impact and political responses? 

**Aim 2**: Can the latent topics emerging from the incident descriptions a) predict countries targeted by cyber attacks and b) societal impact above and beyond the incident types already present in the data set (data theft, disruption, hijacking, physical effects (spatial), physical effects (temporal))?

**Variable codes:** 

weighted_intensity = (data theft + disruption + hijacking + physical effects spatial + physical effect temporal) * effect multiplier [higher for incidents of increasing political importance (critical infrastrucutre, military affected)]
- 1-5 = Low/Moderate Intensity
- 6-10 = High Intensity
- 11-15 = Very High Intensity 

impact_indicator_score = economic impact + political impact + intelligence impact + functional impact 
- 1 = Minor
- 2 = Low
- 3 = Medium
- 4 = High
- 5 = Very High

topic_names = latent cyber operation types extracted from topic modeling (BERTopic)
- Website Defacement & Hacktivism (topic label = 0)
- Malware & Cyber Espionage (topic label = 1)
- Ransomware & Cyber Extortion (topic label = 2)
- DDoS & Service Disruption Campaigns (topic label = 3)
- Data Breaches & Information Exposure (topic label = 4)
- Instiutional Ransomware Attacks (topic label = 5)
- Vulnerability Exploitation & Theft (topic label = 6)
- Surveillance & Mobile Espionage (topic label = 7)
- Government Intelligence Operations (topic label = 8)

topic_groups = condensed clusters based on topic names
- Hacktivism = Website Defacement & Hacktivism
- Disruption Operations = DDoS & Service Disruption Campaigns
- Intrusion = Malware & Cyber Espionage, Vulnerability Exploitation & Theft
- Information = Surveillance & Mobile Espionage, Government Intelligence Operations
- Data Exposure = Data Breaches & Information Exposure
- Financially Motivated Operations = Ransomware & Cyber Extortion, Institutional Ransomware Attacks


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "browser"
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import random
import numpy as np
from src.nlp_functions import(remove_geo_entities)
from scipy.stats import kruskal
import scikit_posthocs as sp
from bertopic import BERTopic
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Load pre-saved model & data 
topic_model = BERTopic.load(
    "data/topic_model/eurepoc_final_model"
)

topic_assignments = pd.read_csv(
    "data/topic_model/eurepoc_topic_assignments.csv"
)

topic_info = pd.read_csv(
    "data/topic_model/topic_info.csv"
)

dyadic_data = pd.read_csv(
    "data/eurepoc_dataset/dyadic_data_topics.csv"
)

In [ ]:
# Topic vs intensity
dyadic_data.groupby("topic_name")["weighted_intensity"]\
    .mean()\
    .sort_values(ascending=False)

In [ ]:
# Topic vs political responses
dyadic_data.groupby("topic_group")["number_political_responses"]\
    .mean()\
    .sort_values(ascending=False)

In [ ]:
# Most common receiver countries by topic
pd.crosstab(
    dyadic_data["receiver_country"],
    dyadic_data["topic_name"]
)

## Visualizations

In [ ]:
# What cyber conflict themes dominate the dataset?
topic_counts = (
    dyadic_data["topic_name"]
    .value_counts()
    .reset_index()
)

topic_counts.columns = ["topic", "count"]

fig = px.treemap(
    topic_counts,
    path=["topic"],
    values="count",
    title="Distribution of Cyber Conflict Topics"
)

fig.show()

fig.write_html("topic_treemap.html")

## Analysis 

The primary goal of this analysis is to investigate what latent categories of cyber operations emerge form inicident descriptions, and answer how these categories are associated with different initiator countries, target countries, and political outcomes. 

Q1: What operational patterns connect initiator countries, cyber operation types, and receiver countries?
- Visualization: Sankey plot 

Q2: Which operational categories are associated with the most severe incidents?
- Visualization: Boxplot of weighted_intenity
- Statistical Test: Kruskal-Wallis

Q3: What types of cyber operations provoke political responses?
- Visualization: Hline plot

Q4: How have trends in cyber operations evolved over time?
- Visualizaiton: Stacked area chart 
- Statistical test: Trend analysis by topic (Spearman's rho)

Q5: Which countries are targeted by which types of cyber operations?
- Visualization: Reciever specialization heatmap

Q6: What cyber operation themes dominate the dataset?
- Visualization: Treemap


### Q1: What operational patterns connect initiator countries, cyber operation types, and receiver countries?

In [ ]:
# Filters for sankey visual
top_initiators = (
    dyadic_data["initiator_country"]
    .value_counts()
    .head(10)
    .index
)

top_receivers = (
    dyadic_data["receiver_country"]
    .value_counts()
    .head(10)
    .index
)

sankey_data = dyadic_data[
    dyadic_data["initiator_country"].isin(top_initiators)
    &
    dyadic_data["receiver_country"].isin(top_receivers)
].copy()

In [ ]:
# Shorten topic labels
sankey_data["short_topic"] = (
    sankey_data["topic_group"]
    .str.split(",")
    .str[:3]
    .str.join(", ")
)

In [ ]:
# Create flows
# Initiator -> Topic
flow1 = (
    sankey_data
    .groupby(
        ["initiator_country", "short_topic"]
    )
    .size()
    .reset_index(name="value")
)

# Topic -> Reciever
flow2 = (
    sankey_data
    .groupby(
        ["short_topic", "receiver_country"]
    )
    .size()
    .reset_index(name="value")
)

In [ ]:
# Build node list
nodes = list(
    pd.concat([
        flow1["initiator_country"],
        flow1["short_topic"],
        flow2["receiver_country"]
    ]).unique()
)

node_dict = {
    node: i
    for i, node in enumerate(nodes)
}

In [ ]:
# Build link table
source = []
target = []
value = []

# Initiator -> Topic
for _, row in flow1.iterrows():
    source.append(
        node_dict[row["initiator_country"]]
    )
    target.append(
        node_dict[row["short_topic"]]
    )
    value.append(row["value"])

# Topic -> Receiver
for _, row in flow2.iterrows():
    source.append(
        node_dict[row["short_topic"]]
    )
    target.append(
        node_dict[row["receiver_country"]]
    )
    value.append(row["value"])

In [ ]:
fig = go.Figure(
    go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            label=nodes
        ),
        link=dict(
            source=source,
            target=target,
            value=value
        )
    )
)

fig.update_layout(
    title="Cyber Conflict Flows: Initiator → Topic → Receiver",
    font_size=10
)

fig.write_html("cyber_conflict_sankey.html")

fig.show()


The Sankey plot visualizes the flow of cyber incidents from initiating actors to operational categories and finally to recipient countries. The width of each flow is proportional to the number of incidents observed in the dataset. Cyber incidents are first groupd by initiator country, then aggregated into five broad operational categories derived from BERTopic analysis: **hacktivism, disruption operations, intrusion operations, information operations, data exposure, and financially motivated operations**. The final stage shows the principal recipient coutnries associated with each operational category. 

**Key Findings**
1. **Financially Motivated operations constitute the largest category of incidents**, indicating that ransomware and extortion-related activities are among the most prevalent forms of cyber conflict represented in the dataset.
2. **Attribution remains a major challenge in cyber conflict**, as "Not Attributed" and "Unkknown" represent some of the largest initiating categories in the diagram.
3. **The US is the largest recipient node**, receiving substantial flows across multiple operational categories, particularily financially motivated operations, intrusion operations, data exposure, and disruption operations. 
4. **Intrusion opeerations are strongly linked sith state-linked actors**. China, Russia, and several other state actors are connected to malware operations, cyber espionage, and vulnerabilty exploitation. The pattern is consistent with the use of cyber capabilities for intelligence collection and network penetration. 

# Q2: Which operational categories are associated with the most impactful incidents?

In [ ]:
topic_summary = (
    dyadic_data
    .groupby("topic_group")
    .agg(
        incidents=("incident_id", "count"),
        avg_impact_score=("impact_indicator_score", "mean"),
        avg_intensity=("weighted_intensity", "mean"),
        avg_political=("number_political_responses", "mean"),
        avg_legal=("number_legal_responses", "mean")
    )
    .round(2)
)

print(topic_summary)

In [ ]:
# Impact indicator score x topic group boxplot
plt.figure(figsize=(12,6))

sns.boxplot(
    data=dyadic_data,
    x="topic_group",
    y="impact_indicator_score"
)

plt.xticks(rotation=90)
plt.ylabel("Impact Indicator Score")
plt.xlabel("Topic Group")
plt.title("Impact Indicator Score Distribution by Operation Type")
plt.show()


In [ ]:
# First drop duplicate incident_ids in the dataset
topic_analysis = (
    dyadic_data
    .drop_duplicates(subset=["incident_id"])
    .copy()
)

# Check sample sizes
print(
    dyadic_data.groupby("topic_group")
    .size()
    .sort_values(ascending=False)
)

# Run Kruskal-Wallis test
groups = [
    g["impact_indicator_score"].dropna()
    for _, g in topic_analysis.groupby("topic_group")
]

stat, p = kruskal(*groups)

print(f"Kruskal-Wallis H-statistic: {stat:.3f}")
print(f"p-value: {p:.6f}")

alpha = 0.05

if p < alpha:
    print("Significant differences in weighted intensity across latent topic groups.")
else:
    print("No significant differences in weighted intensity across latent topic groups.")


In [ ]:
# Dunn test
dunn = sp.posthoc_dunn(
    topic_analysis,
    val_col="impact_indicator_score",
    group_col="topic_group",
    p_adjust="bonferroni"
)


# Convert Dunn matrix to long format
results = []

for i, row in enumerate(dunn.index):
    for j, col in enumerate(dunn.columns):

        # Keep upper triangle only
        if i < j:

            p = dunn.loc[row, col]

            results.append({
                "Comparison": f"{row} vs {col}",
                "p-value": p,
                "Significant": "Yes" if p < 0.05 else "No"
            })

dunn_table = pd.DataFrame(results)

dunn_table = (
    dunn_table
    .sort_values("Comparison")
    .reset_index(drop=True)
)

dunn_table["p-value"] = (
    dunn_table["p-value"]
    .apply(lambda x: "<0.001" if x < 0.001 else f"{x:.3f}")
)

print(dunn_table.to_string(index=False))

## Key Findings

**1. Operational type is associated with severity**
- The Kruskal-Wallis test found significant variation in incident intensity between most categories
- The way a cyber operation is conducted appears to be linked ot its observed societal impact, with ransomware and extortion representing the most distinct category and hacktivist activities generally representing the least severe

**2. Data exposure and financially motivated operations have the largest societal impacts (combination of economic, political, intelligence, and functional effects)**
- Data breaches, ransomware, and cyber extortion are linked to the highest impact scores and display similar profiles

**3. Hacktivism/website defacement occupies the lower-intensity end of the spectrum, but watch for the outliers**
- Website defacements have significantly lower impact scores than all other groups
- But there several cases with unusually high impact scores identified in the boxplot

**4. Disruption operations and information operations show similar impact severity profiles**
- Surveillance, espionage, government intelligence operations, and DDoS campaigns have roughly equal impact scores
- However, it is not clear from this analysis whether they affect different sectors of society in the same way (i.e., economic, political, intelligence, or functional impacts).

**5. Information and intrusion operations are also similar in terms of societal impact**
- Surveillance, espionage, government intelligence operations, malware, vulnerability exploitation and theft have similar impact scores


## Q3: What types of cyber operations provoke political reactions?

In [ ]:
responses = (
    dyadic_data.groupby("topic_group")
    ["number_political_responses"]
    .mean()
    .sort_values()
)

plt.figure(figsize=(10,6))

plt.hlines(
    y=responses.index,
    xmin=0,
    xmax=responses.values
)

plt.plot(
    responses.values,
    responses.index,
    "o"
)

plt.title("Average Political Responses by Topic Group")
plt.show()

In [ ]:
responses = (
    dyadic_data.groupby("topic_name")
    ["number_political_responses"]
    .mean()
    .sort_values()
)

plt.figure(figsize=(10,6))

plt.hlines(
    y=responses.index,
    xmin=0,
    xmax=responses.values
)

plt.plot(
    responses.values,
    responses.index,
    "o"
)

plt.title("Average Political Responses by Topic Names")
plt.show()

## Key Findings:

1. Disruption operations (DDoS and service disruption campaigns) trigger the biggest politic reactions
2. Surviellance and mobile espionage tend to trigger relatively high political reactions, but government intelligence operations trigger none. Noteworthy as both topics are combined under larger "Information Operations" group. 
3. Hacktivism and website defacement are linked with low political responses.
4. Intrusion operations, data exposure, and financially motivated operations occupy the middle of the spectrum. 

## Q4: How have trends in cyber operations evolved over time? 
a) Which types of cyber operations are on the rise/decline?

b) Are cyber operations leading to greater societal impacts over time?

In [ ]:
# Extract year from incident date
dyadic_data["year"] = pd.to_datetime(
    dyadic_data["start_date"],
    errors="coerce"  # invalid dates become NaT
).dt.year

# Calculate annual topic shares
topic_year = pd.crosstab(
    dyadic_data["year"],
    dyadic_data["topic_group"],
    normalize="index"
)

# Consistent color scheme
topic_colors = {
    "Financially Motivated Operations": "#D55E00",  # orange-red
    "Intrusion Operations": "#0072B2",             # blue
    "Data Exposure": "#CC79A7",                    # purple
    "Information Operations": "#009E73",          # green
    "Disruption Operations": "#E69F00",           # amber
    "Hacktivism": "#56B4E9",                      # light blue
}

colors = [
    topic_colors[col]
    for col in topic_year.columns
]

# Plot
fig, ax = plt.subplots(figsize=(14, 7))

topic_year.plot.area(
    ax=ax,
    color=colors
)

ax.set_title("Evolution of Cyber Conflict Topics")
ax.set_xlabel("Year")
ax.set_ylabel("Share of Incidents")

# Move legend outside plot
ax.legend(
    title="Topic",
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

# Leave space for legend
plt.tight_layout(rect=[0, 0, 0.8, 1])

plt.show()

In [ ]:
# Trend analysis by topic
annual_topic_share = (
    pd.crosstab(
        dyadic_data["year"],
        dyadic_data["topic_group"],
        normalize="index"
    )
)

results = []

for topic in annual_topic_share.columns:

    rho, p = spearmanr(
        annual_topic_share.index,
        annual_topic_share[topic]
    )

    results.append([topic, rho, p])

trend_results = pd.DataFrame(
    results,
    columns=["Topic", "Spearman_rho", "p_value"]
)

# Holm correction
trend_results["p_adj"] = multipletests(
    trend_results["p_value"],
    method="holm"
)[1]

trend_results["Significant"] = (
    trend_results["p_adj"] < 0.05
)

trend_results.sort_values("p_adj")


In [ ]:
# Impact over time
annual_impact = (
    dyadic_data
    .groupby("year")["impact_indicator_score"]
    .mean()
)

rho, p = spearmanr(
    annual_impact.index,
    annual_impact.values
)

print(rho, p)

# plot
import seaborn as sns
import matplotlib.pyplot as plt

impact_by_group = (
    dyadic_data
    .groupby(["year", "topic_group"])["impact_indicator_score"]
    .mean()
    .reset_index()
)

# Plot
impact_by_group = (
    dyadic_data
    .groupby(["year", "topic_group"])["impact_indicator_score"]
    .mean()
    .reset_index()
    .sort_values(["topic_group", "year"])
)

# Minimize noisiness of lines
impact_by_group["impact_smoothed"] = (
    impact_by_group
    .groupby("topic_group")["impact_indicator_score"]
    .transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )
)

sns.lineplot(
    data=impact_by_group,
    x="year",
    y="impact_smoothed",
    hue="topic_group",
    palette=topic_colors,
    linewidth=2
)

# Legend on the right
ax.legend(
    title="Operation Category",
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

plt.tight_layout(rect=[0, 0, 0.8, 1])
plt.show()


## Key Findings

1. There has been a sharp uptick in financialy motivated operations in recent years (Spearman's rho = 0.92)
2. Data exposure and disruption operations have also been on the rise. 
3. Intrusion operations have increased since 2000 but have slowed in the last couple of years.
4. The number of hacktivism incidents / website defacements have reduced since 2000. 
5. Information operations have decreased over time. 

In [ ]:
pd.crosstab(
    dyadic_data["receiver_country"],
    dyadic_data["topic_name"],
    normalize="index"
)

In [ ]:
topic_analysis.loc[
    topic_analysis["topic_group"].isna(),
    ["topic_name"]
].drop_duplicates()